In [0]:
events_df = spark.read.table("workspace.ecommerce.ecommerce_delta_table")

events_df.show(5)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+
|2019-11-01 00:00:09|      view|  17501048|2053013558752445019|                NULL|eveline|  7.59|515849878|31e80b9c-e5b3-437...|
|2019-11-01 00:00:28|      view|  17800132|2053013559868129947|   computers.desktop|   NULL| 51.46|529388957|f6835635-57e4-45f...|
|2019-11-01 00:00:37|      view|  21408240|2053013561579406073|  electronics.clocks| tissot|861.28|513118352|4c14bf2a-2820-450...|
|2019-11-01 00:00:43|      view|   5100851|2053013553375346967|                NULL|  honor| 47.62|556727865|4478e949-dcde-412...|
|2019-11-01 00:01:02|      view|   1004720|2053013555631882655|electronics.smart...

In [0]:
from pyspark.sql import functions as F
bronze_table = "workspace.ecommerce.ecommerce_delta_table"
events_df = spark.read.table(bronze_table)

In [0]:
events_df = events_df.filter(
    (F.col("user_id").isNotNull()) &
    (F.col("event_time").isNotNull()) &
    (F.col("price").isNotNull()) &
    (F.col("price") >= 0)
)

In [0]:
features_df = events_df.groupBy("user_id").agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("total_purchases"),
    F.sum("price").alias("total_spent"),
    F.avg("price").alias("avg_price"),
    F.countDistinct("product_id").alias("unique_products"),
    F.max("event_time").alias("last_activity")
)

In [0]:
features_df = features_df.fillna({
    "total_spent": 0,
    "avg_price": 0,
    "total_purchases": 0
})

In [0]:
features_df = features_df.dropDuplicates(["user_id"])

In [0]:
features_df.select("user_id").count()
features_df.select("user_id").distinct().count()

1161642

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.silver_volume
""")

DataFrame[]

In [0]:
silver_path = "/Volumes/workspace/ecommerce/silver_volume/user_features"

features_df.repartition(200) \
    .write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
spark.read.format("delta").load(silver_path) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.user_features_silver")

In [0]:
spark.sql("""
OPTIMIZE workspace.ecommerce.user_features_silver
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("""
OPTIMIZE workspace.ecommerce.user_features_silver
ZORDER BY (user_id)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("""
SELECT user_id, COUNT(*)
FROM workspace.ecommerce.user_features_silver
GROUP BY user_id
HAVING COUNT(*) > 1
""").show()

+-------+--------+
|user_id|COUNT(*)|
+-------+--------+
+-------+--------+



In [0]:
silver_table = "workspace.ecommerce.user_features_silver"

spark.sql(f"""
SELECT
SUM(CASE WHEN total_spent IS NULL THEN 1 ELSE 0 END) AS null_spent,
SUM(CASE WHEN avg_price IS NULL THEN 1 ELSE 0 END) AS null_avg
FROM {silver_table}
""").show()

+----------+--------+
|null_spent|null_avg|
+----------+--------+
|         0|       0|
+----------+--------+



In [0]:
spark.sql("""
SELECT * 
FROM workspace.ecommerce.user_features_silver
LIMIT 10
""").display()

user_id,total_events,total_purchases,total_spent,avg_price,unique_products,last_activity
458172529,2,0,327.44,163.72,2,2019-10-08T05:12:51.000Z
491170114,1,0,180.16,180.16,1,2019-11-26T16:40:29.000Z
510595720,1,0,508.65,508.65,1,2019-11-01T05:29:55.000Z
512377601,2,0,272.34,136.17,1,2019-11-09T11:00:00.000Z
512392370,14,0,1556.52,111.17999999999999,12,2019-11-18T08:29:54.000Z
512410271,3,0,162.76,54.25333333333333,3,2019-10-29T20:15:03.000Z
512426332,1,0,586.63,586.63,1,2019-11-18T11:49:42.000Z
512443190,2,0,646.95,323.475,2,2019-11-02T06:30:53.000Z
512458787,1,0,48.65,48.65,1,2019-10-11T07:48:56.000Z
512475050,1,0,10.3,10.3,1,2019-10-11T20:27:03.000Z
